In [2]:
import streamlit as st
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="901012",
    database="e_commerce"
)

query = """
SELECT
    p.data_pedido,
    pr.nome_produto,
    i.quantidade,
    i.preco_unitario,
    (i.quantidade * i.preco_unitario) AS valor_total
FROM pedidos_silver p
JOIN itens_pedido_silver i ON p.pedido_id = i.pedido_id
JOIN produtos_silver pr ON i.produto_id = pr.produto_id
"""

df = pd.read_sql(query, conn)
df["data_pedido"] = pd.to_datetime(df["data_pedido"])

C:\Users\lucas\AppData\Local\Temp\ipykernel_29912\1991064183.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [3]:
import sys
!{sys.executable} -m pip install mysql-connector-python

  Using cached mysql_connector_python-9.6.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
Using cached mysql_connector_python-9.6.0-cp312-cp312-win_amd64.whl (16.5 MB)


In [3]:
df["ano"] = df["data_pedido"].dt.year
df["mes"] = df["data_pedido"].dt.month

In [4]:
st.title("📊 Dashboard de Vendas")

ano_sel = st.selectbox(
    "Selecione o ano",
    sorted(df["ano"].unique())
)

mes_sel = st.selectbox(
    "Selecione o mês",
    sorted(df[df["ano"] == ano_sel]["mes"].unique())
)

produto_sel = st.multiselect(
    "Selecione o produto",
    df["nome_produto"].unique(),
    default=df["nome_produto"].unique()
)

2026-02-25 12:54:51.882 
  command:

    streamlit run C:\Users\lucas\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-25 12:54:51.893 Session state does not function when running a script without `streamlit run`


In [5]:
df_filtrado = df[
    (df["ano"] == ano_sel) &
    (df["mes"] == mes_sel) &
    (df["nome_produto"].isin(produto_sel))
]

In [6]:
st.subheader("Faturamento por Produto")

faturamento_produto = (
    df_filtrado
    .groupby("nome_produto")["valor_total"]
    .sum()
    .reset_index()
)

st.bar_chart(
    faturamento_produto.set_index("nome_produto")
)

DeltaGenerator()